# Обучение с нуля

Параметры и протокол — `docs/reproduce.md`. Каждый этап запускается отдельно.
Включите GPU, соберите и загрузите `dist/colab_bundle.zip`.
Используется отдельная папка Drive для нового исследования. Тестовая часть не оценивается.


In [ ]:
from pathlib import Path
import io, json, os, shutil, signal, subprocess, sys, zipfile
from google.colab import files, drive

PROJECT = Path('/content/rl_recsys')
PROJECT.mkdir(exist_ok=True)
uploaded = files.upload()  # dist/colab_bundle.zip
assert len(uploaded) == 1
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
    for name in z.namelist():
        allowed = name in {'pyproject.toml', 'constraints_colab.txt'} or name.startswith(('src/', 'tests/', 'configs/', 'scripts/', 'notebooks/'))
        if not allowed or not (PROJECT / name).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError(name)
    z.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(PROJECT / 'constraints_colab.txt'), '-e', str(PROJECT) + '[train,test]'], check=True)
if not Path('/content/drive/MyDrive').exists():
    drive.mount('/content/drive')
ENV = {**os.environ, 'PYTHONPATH': str(PROJECT / 'src'), 'PYTHONUNBUFFERED': '1'}

def run(command):
    with subprocess.Popen(list(map(str, command)), cwd=PROJECT, env=ENV,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            code = process.wait()
        except KeyboardInterrupt:
            if process.poll() is None:
                process.send_signal(signal.SIGINT)
            output, _ = process.communicate(timeout=60)
            print(output or '', end='')
            raise
        if code:
            raise subprocess.CalledProcessError(code, command)

def module(name, *args):
    run([sys.executable, '-m', name, *args])

run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.__version__, torch.cuda.get_device_name())"])
run([sys.executable, '-m', 'pytest', '-q', '-ra'])


## Подготовка и восстановление данных


In [ ]:
BASE = Path('/content/drive/MyDrive/rl_recsys_reproduction')
BASE.mkdir(exist_ok=True)
DATA = PROJECT / 'data'
RUNS = PROJECT / 'runs'
RUNS.mkdir(exist_ok=True)
original = DATA / 'original'
scaled = DATA / 'scaling'
if not original.exists() and (BASE / 'prepared').exists():
    shutil.copytree(BASE / 'prepared', original)
if not scaled.exists() and (BASE / 'scaling_v1/data').exists():
    shutil.copytree(BASE / 'scaling_v1/data', scaled)
if not (original / 'manifest.json').exists() or not (scaled / 'preparation.json').exists():
    if not (DATA / 'raw/interactions.csv').exists():
        module('recsys.cli', 'download', '--output', DATA / 'raw')
    if not (original / 'manifest.json').exists():
        module('recsys.cli', 'prepare', '--input', DATA / 'raw/interactions.csv',
               '--output', original, '--config', 'configs/budget.json',
               '--cutoffs', '2021-07-21', '2021-08-06')
    module('recsys.scaling', '--input', DATA / 'raw/interactions.csv', '--source-data', original,
           '--output', scaled, '--config', 'configs/scaling.json')
shutil.copytree(original, BASE / 'prepared', dirs_exist_ok=True)
shutil.copytree(scaled, BASE / 'scaling_v1/data', dirs_exist_ok=True)
for source, target in [(BASE / 'artifacts', RUNS / 'original'), (BASE / 'scaling_v1', RUNS / 'scaling')]:
    if not target.exists() and source.exists():
        shutil.copytree(source, target, ignore=shutil.ignore_patterns('data'))
print('Данные:', json.loads((scaled / 'large/manifest.json').read_text())['examples'])


## Небольшие запуски и измерение скорости

Эти состояния не используются в основных экспериментах.


In [ ]:
pilot_extra = ['--resume'] if (RUNS / 'pilot/rm_pilot/last.pt').exists() else []
module('recsys.cli', 'train', '--stage', 'rm', '--name', 'rm_pilot', '--data', original,
       '--runs', RUNS / 'pilot', '--backup', BASE / 'pilot', '--max-steps', '12', '--max-minutes', '2', *pilot_extra)
module('recsys.sft_study', '--prepared-root', scaled, '--runs', RUNS / 'scaling',
       '--backup', BASE / 'scaling_v1', '--config', 'configs/scaling.json', '--pilot')
for p in sorted((RUNS / 'scaling/pilot').glob('*/timing.json')):
    print(p.parent.name, json.loads(p.read_text()))


## Обучение модели награды

До 6000 обновлений или 35 минут; выбор по проверочной ошибке.


In [ ]:
extra = ['--resume'] if (RUNS / 'original/rm/last.pt').exists() else []
module('recsys.cli', 'train', '--stage', 'rm', '--data', original,
       '--runs', RUNS / 'original', '--backup', BASE / 'artifacts', *extra)


## Сравнение объема данных и размера GPT-2

Три ограниченных запуска, до 45 минут каждый.


In [ ]:
module('recsys.sft_study', '--prepared-root', scaled, '--runs', RUNS / 'scaling',
       '--backup', BASE / 'scaling_v1', '--config', 'configs/scaling.json')


## Серии RL

Выполните блок для `mods`, `preserve`, затем `ablation`, меняя STUDY.
Сначала подготовка и небольшой запуск; основной запуск — в следующей ячейке.
Границы времени сохраняются в журнале эксперимента.


In [ ]:
STUDY = 'preserve'  # mods / preserve / ablation
NAME = {'mods': 'rl_mods_v1', 'preserve': 'rl_preserve_v1', 'ablation': 'rl_ablation_v1'}[STUDY]
config = RUNS / (NAME + '.json')
sft = RUNS / 'scaling/large_128/best.pt'
rm = RUNS / 'original/rm/best.pt'
run([sys.executable, 'scripts/bind_run.py', '--study', STUDY, '--data', scaled / 'large',
     '--sft', sft, '--rm', rm, '--output', config])
args = ['--data', scaled / 'large', '--runs', RUNS / NAME, '--backup', BASE / NAME,
        '--source-sft', sft, '--source-rm', rm, '--reward-source-data', original, '--config', config]
module('recsys.rl_study', '--stage', 'prepare', *args)
module('recsys.rl_study', '--stage', 'pilot', *args)


In [ ]:
module('recsys.rl_study', '--stage', 'train', *args)


## Фиксация до тестовой оценки

После завершения `preserve` и `ablation` фиксируем состояния нового обучения.
Для опубликованных чисел используется `configs/final_selection.json`;
новая фиксация хранится отдельно и не заменяет результаты исследования.


In [ ]:
run([sys.executable, 'scripts/freeze_run.py', '--base', BASE,
     '--output', RUNS / 'fresh_selection.json'])
shutil.copy2(RUNS / 'fresh_selection.json', BASE / 'fresh_selection.json')
